# UVERA · NB01 — AI-1 Pause Check

| | |
|---|---|
| **Accelerator** | None (CPU) |
| **Internet** | On |
| **Secret** | — |
| **Inputs to add** | — |
| **Expected runtime** | ~20–40 min |
| **How to run** | **Save Version → Save & Run All (Commit)** — runs in the background, no 20-minute idle timeout, 12-hour cap |

**What this AI does (say this to judges):** Before a customer taps Confirm, AI-1 estimates the chance that the transfer is a scam, says how sure it is, and gives the top reasons. It is compared with the simple rule a basic fraud system uses (big amount + new receiver). When the evidence is mixed, it says 'not sure' and a person checks, instead of guessing.

**Outputs** (in `/kaggle/working/outputs/NB01_outputs.zip`): `artifacts/ai1/` (model.txt, calibrator.json, conformal.json, novelty.json, thresholds.json, test_scores.parquet), `reports/metrics_ai1.json`, figures

All data is synthetic or open-licensed. No real customer data.

In [ ]:
# 1) Get the UVERA code from the public GitHub repo (nothing to upload by hand)
REPO = "https://github.com/ABRUBAB/takaresolve-ai-hackathon-2026.git"
REF = "main"   # replace with a commit hash to reproduce an exact run
import os, sys, subprocess
if not os.path.exists("/tmp/uvera"):
    subprocess.run(["git", "clone", "-q", REPO, "/tmp/uvera"], check=True)
subprocess.run(["git", "-C", "/tmp/uvera", "fetch", "-q", "origin", REF], check=False)
subprocess.run(["git", "-C", "/tmp/uvera", "checkout", "-q", REF], check=True)
COMMIT = subprocess.run(["git", "-C", "/tmp/uvera", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.environ["UVERA_ROOT"] = "/tmp/uvera"
sys.path.insert(0, "/tmp/uvera/ml")   # use Kaggle's own numpy/pandas/sklearn/lightgbm (no reinstall)
print("UVERA code at commit", COMMIT)
print(f'To reproduce this exact run later, set REF = "{COMMIT}" in this cell.')

In [ ]:
# 2) Reproducibility: seeds, versions, time budget, output folder
import json
from IPython.display import Image, display
from uvera_ml.common import env_info, set_seed, Budget, out_dir, zip_outputs, write_json
set_seed(42)
budget = Budget(minutes=150)
OUT = out_dir("NB01")
info = env_info(); info["uvera_commit"] = COMMIT
write_json(OUT / "reports" / "versions_nb01.json", info)
print(json.dumps(info, indent=1))

In [ ]:
# 3) Build the synthetic world. Same seed -> identical data on every Kaggle account (checked by hash).
from pathlib import Path
from uvera_ml.sim.world import load_or_generate
world = load_or_generate("/tmp/world_full", scale="full")
print(json.dumps(world.meta, indent=1)[:1500])
ref = Path("/tmp/uvera/reports/world_meta.json")
if ref.exists():
    expected = json.loads(ref.read_text(encoding="utf-8"))["meta"]["hashes"]["events"]
    print("Data identical to NB00 run:", expected == world.meta["hashes"]["events"])
budget.check("world")

## Train, calibrate, test once
Steps: rule baseline → logistic regression → LightGBM with **StratifiedGroupKFold (5 folds, grouped by customer) × 5 seeds**,
plus XGBoost and CatBoost for comparison → isotonic calibration → Mondrian conformal ("not sure") → novelty flag →
thresholds frozen on validation → **one** pass on the test set (customers never seen in training).

In [ ]:
from uvera_ml.models.ai1 import run_ai1
summary = run_ai1(world, OUT, seeds=(42, 7, 1337, 2026, 99))
t = summary["test_unseen_customers"]
print(json.dumps({"model": t["model"], "rule_baseline": t["rule_baseline"], "conformal": t["conformal"],
                  "unsure_rate": t["unsure_rate"], "selective": t["selective"]}, indent=1, default=float))
budget.check("ai1")

## Results
Calibration, precision-recall vs the rule baseline, error vs coverage (abstention), and what drives the model.

In [ ]:
for f in ["ai1_reliability.png", "ai1_pr_curve.png", "ai1_risk_coverage.png", "ai1_shap_global.png"]:
    display(Image(str(OUT / "reports" / "figures" / f)))
import pandas as pd
display(pd.DataFrame(summary["cv_summary"]))
display(pd.DataFrame(summary["ablation_validation"]))
display(pd.DataFrame(summary["fairness_slices_test_window"]))

In [ ]:
# Last step: package the results. Download NB01_outputs.zip from the Output tab.
budget.check("done")
zip_outputs(OUT)